# 08. Nodos centrales y participantes puente

Se retoman la red bipartita autor-video del punto 4 y sus proyecciones del punto 5 para identificar
qué autores y videos ocupan posiciones estructuralmente centrales. El punto 6 ya mostró que la red
es casi un árbol muy fragmentado (10 componentes, 97% de aristas puente en la componente mayor) y
el punto 7 encontró que casi todas las comunidades corresponden a un único video. Este punto retoma
esos hallazgos para señalar, con medidas de centralidad, cuáles nodos concretos sostienen esa
estructura y cuáles quedan si se eliminan.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

def localizar(nombre, subcarpeta):
    candidatos = [Path.cwd() / subcarpeta / nombre, Path.cwd() / nombre,
                  Path.cwd().parent / subcarpeta / nombre]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Ejecute primero los notebooks previos.")

nodos_bip = pd.read_csv(localizar("04_nodos_bipartita.csv", "salidas"), dtype={"raw_id": "string"})
aristas_bip = pd.read_csv(localizar("04_aristas_bipartita.csv", "salidas"))
nodos_aa = pd.read_csv(localizar("05_nodos_autor_autor.csv", "salidas"), dtype={"author_channel_id": "string"})
aristas_aa = pd.read_csv(localizar("05_aristas_autor_autor.csv", "salidas"))
nodos_vv = pd.read_csv(localizar("05_nodos_video_video.csv", "salidas"), dtype={"video_id": "string"})
aristas_vv = pd.read_csv(localizar("05_aristas_video_video.csv", "salidas"))
comunidades_nodos = pd.read_csv(localizar("07_comunidades_nodos.csv", "salidas"))

B = nx.Graph()
B.add_nodes_from(nodos_bip.loc[nodos_bip["node_type"] == "author", "node_id"], bipartite=0)
B.add_nodes_from(nodos_bip.loc[nodos_bip["node_type"] == "video", "node_id"], bipartite=1)
B.add_weighted_edges_from(aristas_bip[["source", "target", "weight"]].itertuples(index=False, name=None))

G_aa = nx.Graph()
G_aa.add_nodes_from(nodos_aa["author_channel_id"])
G_aa.add_weighted_edges_from(aristas_aa[["source", "target", "weight"]].itertuples(index=False, name=None))

G_vv = nx.Graph()
G_vv.add_nodes_from(nodos_vv["video_id"])
G_vv.add_weighted_edges_from(aristas_vv[["source", "target", "weight"]].itertuples(index=False, name=None))

AUTORES = set(nodos_bip.loc[nodos_bip["node_type"] == "author", "node_id"])
VIDEOS = set(nodos_bip.loc[nodos_bip["node_type"] == "video", "node_id"])
info_nodo = nodos_bip.set_index("node_id")
comunidad_de = comunidades_nodos.set_index("node_id")["comunidad"].to_dict()

print(f"Bipartita: {B.number_of_nodes()} nodos ({len(AUTORES)} autores, {len(VIDEOS)} videos), {B.number_of_edges()} aristas")
print(f"Autor-autor: {G_aa.number_of_nodes()} nodos, {G_aa.number_of_edges():,} aristas")
print(f"Video-video: {G_vv.number_of_nodes()} nodos, {G_vv.number_of_edges()} aristas")


Bipartita: 351 nodos (332 autores, 19 videos), 343 aristas
Autor-autor: 332 nodos, 10,732 aristas
Video-video: 19 nodos, 11 aristas


## 8.1 Elección y justificación de las medidas de centralidad

El grado y la fuerza ya se calcularon en los puntos 5 y 6 y sirven como medida de
alcance directo. Aquí se añaden tres medidas que capturan roles distintos al de solo alcance:

**Betweenness (intermediación).** Mide en qué proporción de los caminos más cortos entre pares de
nodos aparece un nodo dado. Es la medida más directamente ligada a lo que se pide, ya que un autor o video
con betweenness alto es, por definición, un paso obligado entre partes de la red que de otro modo
quedarían separadas. Se calcula con peso, ya que la distancia de una arista se define como el inverso de
su peso, de modo que vínculos más fuertes cuentan como más cercanos.

**PageRank.** PageRank no requiere queb la red sea conexa, ya que su mecanismo de reinicio aleatorio le da un valor bien definido a todo
nodo aun cuando la red tiene 10 componentes, siendo ese el caso. Por eso se prefiere sobre eigenvector
centrality, cuyo valor colapsa a 0 en los componentes que no contienen al autovalor dominante de la
red completa, es decir, en 9 de los 10 componentes de esta red. PageRank además pondera por el peso
de la arista, lo que aquí equivale a favorecer a los videos con más comentarios por autor y no solo
más autores distintos.

**Closeness o cercanía.** Tiene sentido dentro de un componente conexo, así que se calcula
únicamente sobre la componente mayor, 286 de 351 nodos siendo 81.5%, donde los 65 nodos restantes, en 9
componentes menores, no tienen un valor comparable y se excluyen de esa columna sin convertirlos en
ceros, para no simular una cercanía que no existe.

No se reporta una eigenvector centrality independiente de PageRank porque sobre una red casi arbórea y
desconectada como esta, el método de potencias que la calcula converge mal o no converge, y el
resultado depende fuertemente de qué componente se analice. PageRank resuelve ese problema y se usa
como la versión utilizable de la misma idea.

In [2]:
Bw = B.copy()
for _, _, d in Bw.edges(data=True):
    d["distancia"] = 1.0 / d["weight"]

betweenness = nx.betweenness_centrality(Bw, weight="distancia", normalized=True)
pagerank = nx.pagerank(B, weight="weight")

componentes_bip = sorted(nx.connected_components(B), key=len, reverse=True)
gigante = B.subgraph(componentes_bip[0])
closeness_gigante = nx.closeness_centrality(gigante)

centralidad = pd.DataFrame({"node_id": list(B.nodes())})
centralidad["tipo"] = centralidad["node_id"].map(lambda n: "autor" if n in AUTORES else "video")
centralidad["etiqueta"] = centralidad["node_id"].map(info_nodo["label"])
centralidad["grado"] = centralidad["node_id"].map(dict(B.degree()))
centralidad["en_componente_mayor"] = centralidad["node_id"].isin(componentes_bip[0])
centralidad["betweenness"] = centralidad["node_id"].map(betweenness)
centralidad["pagerank"] = centralidad["node_id"].map(pagerank)
centralidad["closeness_comp_mayor"] = centralidad["node_id"].map(closeness_gigante)

print(f"Componente mayor: {len(componentes_bip[0])} de {B.number_of_nodes()} nodos "
      f"({100*len(componentes_bip[0])/B.number_of_nodes():.1f}%); closeness solo se calcula ahí.")
print(f"\nTop 8 por betweenness (rol de puente)")
print(centralidad.sort_values("betweenness", ascending=False).head(8)
      [["node_id", "tipo", "etiqueta", "grado", "betweenness"]].to_string(index=False))
print(f"\nTop 8 por PageRank (importancia ponderada por comentarios)")
print(centralidad.sort_values("pagerank", ascending=False).head(8)
      [["node_id", "tipo", "etiqueta", "grado", "pagerank"]].to_string(index=False))


Componente mayor: 286 de 351 nodos (81.5%); closeness solo se calcula ahí.

Top 8 por betweenness (rol de puente)
                         node_id  tipo                                                    etiqueta  grado  betweenness
              video::n8iP75gIpmw video                                   Qué rico come tu diputado    128        0.565
              video::PjmxCj-a9Hg video Conferencia de Prensa del Gobierno de Guatemala. #LaRondaGt     19        0.244
author::UCHTGCgY2l-DQJIvlA_cpa_Q autor                                         @virgiliogarcia3039      2        0.232
              video::j43HgwYFKfk video               La cooptación de Walter Mazariegos en la USAC     49        0.228
              video::6W4u8sGEnGM video  Inician los trabajos de recuperación del Puente Belice II.     32        0.188
author::UCylqlpsh8ENNM1LqgQ1KbxA autor                                                @Jel.Awesh.M      2        0.178
author::UCzZ6dDCEsLMXbc5pCFXIprw autor               

Los cinco primeros lugares de betweenness son los mismos videos que el punto 6 señaló como el
esqueleto de la componente mayor, más un autor puntual, `@virgiliogarcia3039`. El video "Qué rico
come tu diputado" concentra un betweenness de 0.565. Esto muestra que más de la mitad de los caminos más cortos entre
pares de nodos de la red pasan por él. En PageRank domina el mismo video (0.167), seguido de "La
cooptación de Walter Mazariegos en la USAC" (0.063); ambos coinciden con los videos de mayor alcance
ya vistos en el punto 6, lo que es esperable porque PageRank pondera el peso de la arista y aquí el
peso principal proviene del volumen de comentarios por video.

## 8.2 Interpretación por separado: autores y videos

**Autores: recurrencia y diversidad de participación.** El grado en la red bipartita no tiene
sentido de "popularidad" para un autor: mide en cuántos videos distintos comentó. Solo 9 de los 332
autores (2.7%) comentaron en más de un video, y son justamente los candidatos a autor puente que
adelantó el punto 3.5. Se listan junto con su betweenness, para distinguir entre recurrencia simple
(comentar en más de un video) y recurrencia que además conecta partes distintas de la red
(betweenness alto).

In [3]:
autores_tabla = nodos_bip[nodos_bip["node_type"] == "author"][
    ["node_id", "label", "comment_count", "videos_commented", "channels_commented"]
].copy()
autores_tabla["betweenness"] = autores_tabla["node_id"].map(betweenness)
autores_tabla["pagerank"] = autores_tabla["node_id"].map(pagerank)
autores_tabla["articulacion"] = autores_tabla["node_id"].isin(nx.articulation_points(gigante))

recurrentes = autores_tabla[autores_tabla["videos_commented"] > 1].sort_values(
    "betweenness", ascending=False
)
print(f"Autores con más de un video comentado: {len(recurrentes)} de {len(autores_tabla)} "
      f"({100*len(recurrentes)/len(autores_tabla):.1f}%)")
recurrentes[["node_id", "label", "comment_count", "videos_commented", "channels_commented",
             "betweenness", "articulacion"]]


Autores con más de un video comentado: 9 de 332 (2.7%)


,node_id,label,comment_count,videos_commented,channels_commented,betweenness,articulacion
98,author::UCHTGCgY2l-DQJIvlA_cpa_Q,@virgiliogarcia3039,3,2.000,2.000,0.232,True
325,author::UCylqlpsh8ENNM1LqgQ1KbxA,@Jel.Awesh.M,3,2.000,1.000,0.178,False
330,author::UCzZ6dDCEsLMXbc5pCFXIprw,@josegil3813,2,2.000,1.000,0.177,True
276,author::UCpsKOkt5iWTbenzeuq7dsmQ,@inge_vergueta,3,3.000,1.000,0.129,True
307,author::UCvcu1kR8xMYy_I1Ty-2mV2Q,@hashojea7348,4,3.000,1.000,0.092,True
250,author::UCjZgFEowMBrsjodqfovzdKw,@franciscoflores3120,2,2.000,2.000,0.058,True
211,author::UCdFlugHJJa4l3YqWuNRmvXw,@MarcosCarillo-b1r,2,2.000,2.000,0.032,True
287,author::UCsUCN0Yq_UyTvKjOJLmNrnQ,@moisesvaldez4043,2,2.000,2.000,0.032,True
207,author::UCbrtvygfRT6QXqWDNrOf-cA,@Alejandro00710,2,2.000,1.000,0.000,False


Los 9 autores recurrentes representan diversidad de canales de forma desigual, por ejemplo `@inge_vergueta` y
`@hashojea7348` comentaron 3 videos pero dentro de un solo canal (Quorum), mientras que
`@virgiliogarcia3039`, `@franciscoflores3120`, `@MarcosCarillo-b1r` y `@moisesvaldez4043` cruzan dos
canales distintos con solo 2 videos. La diversidad de canales, no el número de videos, es lo que más
se asocia con betweenness alto. La excepción es `@Jel.Awesh.M`, que solo comenta dentro de Quorum y aun así tiene el sexto betweenness más alto de
toda la red, esto ya que sus dos videos son "Internet: escoger el menos malo" y otro video menor, pero ese vínculo
resulta ser el único paso hacia una parte de la red que de otro modo quedaría aparte.

**Videos: alcance y capacidad de conectar audiencias.** Para un video, el grado bipartito
(autores únicos) mide alcance de la muestra recolectada, no de la audiencia real del video. La
capacidad de conectar audiencias distintas es otra cosa, ya que se mide mejor en la proyección
video-video del punto 5, donde una arista indica que al menos un autor comentó en ambos videos. Se
añade también el betweenness y la condición de articulador en cada una de las dos redes.

In [4]:
videos_tabla = nodos_bip[nodos_bip["node_type"] == "video"][
    ["node_id", "label", "channel_name", "unique_authors", "comment_count", "view_count"]
].copy()
videos_tabla["betweenness_bipartita"] = videos_tabla["node_id"].map(betweenness)
videos_tabla["pagerank"] = videos_tabla["node_id"].map(pagerank)
videos_tabla["articulador_bipartita"] = videos_tabla["node_id"].isin(nx.articulation_points(gigante))

Gw_vv = G_vv.copy()
for _, _, d in Gw_vv.edges(data=True):
    d["distancia"] = 1.0 / d["weight"]
betweenness_vv = nx.betweenness_centrality(Gw_vv, weight="distancia")
gigante_vv = G_vv.subgraph(max(nx.connected_components(G_vv), key=len))
articuladores_vv = set(nx.articulation_points(gigante_vv))

videos_tabla["video_id"] = videos_tabla["node_id"].str.replace("video::", "", regex=False)
videos_tabla["betweenness_video_video"] = videos_tabla["video_id"].map(betweenness_vv)
videos_tabla["grado_video_video"] = videos_tabla["video_id"].map(dict(G_vv.degree()))
videos_tabla["articulador_video_video"] = videos_tabla["video_id"].isin(articuladores_vv)

videos_tabla = videos_tabla.sort_values("unique_authors", ascending=False)
videos_tabla[["label", "channel_name", "unique_authors", "comment_count", "pagerank",
              "betweenness_bipartita", "articulador_bipartita",
              "grado_video_video", "betweenness_video_video", "articulador_video_video"]]


,label,channel_name,unique_authors,comment_count,pagerank,betweenness_bipartita,articulador_bipartita,grado_video_video,betweenness_video_video,articulador_video_video
346,Qué rico come tu diputado,Quorum,128.000,161,0.167,0.565,True,4,0.150,True
344,La cooptación de Walter Mazariegos en la USAC,Quorum,49.000,50,0.063,0.228,True,3,0.052,True
334,Inician los trabajos de recuperación del Puente Belice II.,Gobierno de la República de Guatemala,32.000,45,0.043,0.188,True,2,0.052,True
345,Plan 2032 Ciudad de Guatemala,Municipalidad de Guatemala,25.000,25,0.034,0.005,False,0,0.000,False
337,Conferencia de Prensa del Gobierno de Guatemala. #LaRondaGt,Gobierno de la República de Guatemala,19.000,25,0.025,0.244,True,2,0.092,True
336,EE.UU. envía a mexicanos deportados a Guatemala antes de su regres...,Noticias Telemundo,18.000,25,0.025,0.003,False,0,0.000,False
349,Arroz con pollo a la MONOPOLIO,Quorum,16.000,16,0.020,0.064,True,2,0.000,False
332,Capturan a presuntos delincuentes disfrazados de mujer señalados d...,Noti7,13.000,14,0.018,0.055,True,1,0.000,False
347,Internet: escoger el menos malo,Quorum,10.000,12,0.013,0.128,True,4,0.092,True
343,Capturan a ladrón que había quedado grabado mientras robaba en una...,TN23 Guatemala,7.000,7,0.010,0.028,True,1,0.000,False


"Qué rico come tu diputado" combina el mayor alcance (128 autores únicos) con el mayor grado en la
proyección video-video y el mayor betweenness en ambas redes, por tanto es a la vez el video más visto por autores distintos y el más conectado a otros contenidos. Nótese
el contraste con "Plan 2032 Ciudad de Guatemala" que tiene 25 autores, tercer lugar en alcance y "EE.UU.
envía a mexicanos deportados..." con 18 autores, ya que ambos tienen grado 0 en la proyección video-video,
es decir, ninguno de sus comentaristas apareció en otro video de la muestra. Alcance alto no implica
capacidad de conectar audiencias, estas son dimensiones distintas y en esta muestra se disocian en varios
videos. Los cinco videos articuladores de la proyección video-video coinciden exactamente con los cinco videos articuladores
de la red bipartita, lo que es consistente porque ambas redes describen la misma evidencia desde
ángulos distintos.

## 8.3 Participantes recurrentes, autores puente y videos articuladores

El punto 6 ya había contado 17 puntos de articulación en la componente mayor sin listarlos. Aquí se
identifican por nombre y se distingue explícitamente entre autor recurrente, que comenta en más de un
video y autor puente. De los 9 autores recurrentes, 7 son puntos de articulación, mientras que los otros 2, `@Jel.Awesh.M` y `@Alejandro00710`, no lo son, lo que solo
puede significar que existe al menos otra ruta que conecta a los mismos videos sin pasar por ellos.

In [5]:
puentes_autores = recurrentes[recurrentes["articulacion"]]["node_id"].tolist()

conteo_antes = nx.number_connected_components(B)
gigante_antes = len(max(nx.connected_components(B), key=len))

G_sin_puentes = gigante.copy()
G_sin_puentes.remove_nodes_from(puentes_autores)
fragmentos = sorted(nx.connected_components(G_sin_puentes), key=len, reverse=True)

print(f"Componentes de la red completa: {conteo_antes}; componente mayor: {gigante_antes} nodos.")
print(f"\nAl eliminar los {len(puentes_autores)} autores puente (los recurrentes que además son "
      f"puntos de articulación) SOLO del componente mayor:")
print(f"  Componente mayor pasa de {gigante.number_of_nodes()} a {len(fragmentos[0])} nodos.")
print(f"  Aparecen {len(fragmentos)} fragmentos en lugar de 1.")
for i, frag in enumerate(fragmentos):
    videos_frag = sorted(info_nodo.loc[[n for n in frag if n in VIDEOS], "label"].tolist())
    print(f"   Fragmento {i+1}: {len(frag)} nodos -> {videos_frag}")


Componentes de la red completa: 10; componente mayor: 286 nodos.

Al eliminar los 7 autores puente (los recurrentes que además son puntos de articulación) SOLO del componente mayor:
  Componente mayor pasa de 286 a 173 nodos.
  Aparecen 8 fragmentos en lugar de 1.
   Fragmento 1: 173 nodos -> ['La cooptación de Walter Mazariegos en la USAC', 'Qué rico come tu diputado']
   Fragmento 2: 31 nodos -> ['Inician los trabajos de recuperación del Puente Belice II.']
   Fragmento 3: 24 nodos -> ['Arroz con pollo a la MONOPOLIO', 'Internet: escoger el menos malo']
   Fragmento 4: 18 nodos -> ['Conferencia de Prensa del Gobierno de Guatemala. #LaRondaGt']
   Fragmento 5: 13 nodos -> ['Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer asalto']
   Fragmento 6: 7 nodos -> ['Bloqueos en Guatemala este 31 de agosto por alza en combustibles afectan rutas principales']
   Fragmento 7: 7 nodos -> ['Capturan a ladrón que había quedado grabado mientras robaba en una parroquia de 

Eliminar simultáneamente a los 7 autores puente reduce la componente mayor de 286 a 173 nodos y la
parte en 8 fragmentos en lugar de 1. El fragmento más grande, 173 nodos, todavía conserva unidos
"Qué rico come tu diputado" y "La cooptación de Walter Mazariegos en la USAC", aunque
`@inge_vergueta` fue eliminada. `@Alejandro00710` y `@Jel.Awesh.M` son los dos recurrentes que no son puntos de articulación, porque proveen una ruta alterna entre
esos dos videos específicos. Esa es la única redundancia real de conexión en toda la componente mayor,
en el resto, cada corte de un autor puente separa un contenido completo del resto de la red, tal
como anticipaba el 97% de aristas puente del punto 6.

In [6]:
SALIDAS = Path.cwd() / "salidas"
SALIDAS.mkdir(exist_ok=True)

autores_tabla.to_csv(SALIDAS / "08_centralidad_autores.csv", index=False)
videos_tabla.drop(columns=["node_id"]).to_csv(SALIDAS / "08_centralidad_videos.csv", index=False)

puentes_export = pd.concat([
    recurrentes.assign(rol="autor_puente")[["node_id", "label", "betweenness", "articulacion", "rol"]],
    videos_tabla[videos_tabla["articulador_bipartita"]].assign(rol="video_articulador")
        .rename(columns={"betweenness_bipartita": "betweenness", "articulador_bipartita": "articulacion"})
        [["node_id", "label", "betweenness", "articulacion", "rol"]],
], ignore_index=True)
puentes_export.to_csv(SALIDAS / "08_puentes_articuladores.csv", index=False)
print("Centralidad y puentes exportados en ./salidas/")


Centralidad y puentes exportados en ./salidas/


## Conclusión del punto 8

La red no tiene un centro difuso sino un puñado de nodos concretos que sostienen su conectividad, asi como
un video que por sí solo explica más de la mitad de los caminos más cortos de toda la red y 7 autores que aparecen en dos o tres videos cada uno y cuya eliminación
fragmenta la componente mayor en 8 pedazos, con solo un par de videos protegido por una ruta
redundante. El alcance de un video con autores únicos y su capacidad de conectar audiencias
distintas no son la misma propiedad, ya que varios videos con alcance considerable no comparten ningún
autor con otro video de la muestra. Estos hallazgos describen la topología de los 406 comentarios
recolectados y no permiten inferir coordinación, conversación ni influencia entre los autores
identificados como puente.